<a href="https://colab.research.google.com/github/DenGodunov/AirflowProject/blob/main/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D0%B5%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [147]:
# Установка необходимой библиотеки для подключения к базе данных ClickHouse
!pip install clickhouse-driver  # Установка Python-коннектора для ClickHouse

In [148]:
# Импортируем необходимые библиотеки
import requests as req  # для выполнения HTTP-запросов
import pandas as pd  # для обработки данных
from datetime import datetime, timedelta  # для работы с датами
import json  # для парсинга json
from clickhouse_driver import Client  # для подключения к ClickHouse
import ast

In [149]:
# Устанавливаем URL для API и начальную дату для извлечения данных
URL = 'https://api.exchangerate.host/timeframe'  # URL для получения курсов валют с сайта ЦБ
ACCESS_KEY = '043dc9dad696914726d3064e9d917294'
SOURCE = 'USD'
START_DATE = '2023-01-01'  # Дата для получения данных по валютам
END_DATE = '2023-01-01'  # Дата для получения данных по валютам

In [150]:
# Настройка подключения к базе данных ClickHouse
CH_CLIENT = Client(
    host='158.160.116.58',  # IP-адрес сервера ClickHouse
    user='student',  # Имя пользователя для подключения
    password='dfqh89fhq8',  # Пароль для подключения
    database='sandbox'  # База данных, к которой подключаемся
)

In [151]:
# Функция для извлечения данных с API и сохранения их в локальный файл
def extract_data(url, s_file, access_key, source, start_date, end_date):
    """
    Эта функция выгружает данные по валютам, используя GET-запрос,
    и сохраняет результат в локальный файл `s_file`.
    """
    print(f"{URL}?access_key={ACCESS_KEY}&source={SOURCE}&start_date={START_DATE}&end_date={END_DATE}")
    request = req.get(f"{URL}?access_key={ACCESS_KEY}&source={SOURCE}&start_date={START_DATE}&end_date={END_DATE}")  # Выполняем GET-запрос для получения данных за указанную дату
    #print(url + '?' + access_key +'&'+ source + '&' + start_date + '&' + end_date)

    # Сохраняем полученные данные (в формате JSON) в локальный файл
    with open(s_file, "w", encoding="utf-8") as tmp_file:
        print(tmp_file)
        tmp_file.write(request.text)  # Записываем текст ответа в файл




In [137]:
#print(f"{URL}?access_key={ACCESS_KEY}&source={SOURCE}&start_date={START_DATE}&end_date={END_DATE}")


In [152]:
# Функция для обработки данных в формате JSON и преобразования их в CSV
def transform_data(s_file, csv_file, date):
    """
    Эта функция обрабатывает полученные данные в формате XML
    и преобразует их в CSV формат для дальнейшей работы.
    """
    rows = list()
    with open('currency.json', 'r', encoding='utf-8') as file:
    # Метод load считывает данные и превращает их в структуру Python
      data = json.load(file)
      quotes = data["quotes"]
      obj = quotes["2023-01-01"]
      #print(obj)
    # Перебор всех валют в XML и извлечение нужных данных
      for currency, rate in obj.items():

        rows.append((currency,rate))

      print(rows)
      data_frame = pd.DataFrame(
        rows, columns=["currency", "rate"]
    )
      #print(data_frame)
      data_frame['date'] = START_DATE
      print(data_frame)
      data_frame.to_csv(csv_file, sep=",", encoding="utf-8", index=False)









In [153]:
transform_data('currency', 'currency.csv',START_DATE)

[('USDAED', 3.672635), ('USDAFN', 87.49408), ('USDALL', 107.150283), ('USDAMD', 393.731324), ('USDANG', 1.802385), ('USDAOA', 503.690999), ('USDARS', 176.728445), ('USDAUD', 1.46638), ('USDAWG', 1.8), ('USDAZN', 1.700045), ('USDBAM', 1.832031), ('USDBBD', 2.019174), ('USDBDT', 103.159191), ('USDBGN', 1.827155), ('USDBHD', 0.37658), ('USDBIF', 2062), ('USDBMD', 1), ('USDBND', 1.34029), ('USDBOB', 6.910305), ('USDBRL', 5.286699), ('USDBSD', 1.000033), ('USDBTC', 6.0217375e-05), ('USDBTN', 82.580068), ('USDBWP', 12.755638), ('USDBYN', 2.524225), ('USDBYR', 19600), ('USDBZD', 2.01584), ('USDCAD', 1.353395), ('USDCDF', 2029.99989), ('USDCHF', 0.92395), ('USDCLF', 0.03075), ('USDCLP', 848.250029), ('USDCNY', 6.897897), ('USDCNH', 6.92034), ('USDCOP', 4848.67), ('USDCRC', 591.730443), ('USDCUC', 1), ('USDCUP', 26.5), ('USDCVE', 103.298862), ('USDCZK', 22.52401), ('USDDJF', 177.719972), ('USDDKK', 6.949525), ('USDDOP', 56.25032), ('USDDZD', 137.17938), ('USDEGP', 24.752919), ('USDERN', 15), ('

In [154]:
# Функция для загрузки данных в ClickHouse из CSV
def upload_to_clickhouse(csv_file, table_name, client):
    """
    Эта функция считывает CSV файл, создает таблицу в
    базе данных ClickHouse и добавляет данные в неё
    """
    # Чтение данных из CSV
    data_frame = pd.read_csv(csv_file)

    # Создание таблицы, ЕСЛИ НЕ СУЩЕСТВУЕТ ТО СОЗДАТЬ ТАБЛИЦУ
    client.execute(f'CREATE TABLE IF NOT EXISTS {table_name} (currency String, rate Float, date String) ENGINE Log')

    # Запись data frame в ClickHouse
    client.execute(f'INSERT INTO {table_name} VALUES', data_frame.to_dict('records'))

In [155]:
extract_data(URL, 'currency.json', ACCESS_KEY, SOURCE, START_DATE,END_DATE)
transform_data('currency.json', 'currency.csv', START_DATE)  # Преобразование данных в формат CSV
upload_to_clickhouse(
    csv_file='currency.csv',
    table_name='currency_data_godunov_denis',
    client=CH_CLIENT
)

https://api.exchangerate.host/timeframe?access_key=043dc9dad696914726d3064e9d917294&source=USD&start_date=2023-01-01&end_date=2023-01-01
<_io.TextIOWrapper name='currency.json' mode='w' encoding='utf-8'>
[('USDAED', 3.672635), ('USDAFN', 87.49408), ('USDALL', 107.150283), ('USDAMD', 393.731324), ('USDANG', 1.802385), ('USDAOA', 503.690999), ('USDARS', 176.728445), ('USDAUD', 1.46638), ('USDAWG', 1.8), ('USDAZN', 1.700045), ('USDBAM', 1.832031), ('USDBBD', 2.019174), ('USDBDT', 103.159191), ('USDBGN', 1.827155), ('USDBHD', 0.37658), ('USDBIF', 2062), ('USDBMD', 1), ('USDBND', 1.34029), ('USDBOB', 6.910305), ('USDBRL', 5.286699), ('USDBSD', 1.000033), ('USDBTC', 6.0217375e-05), ('USDBTN', 82.580068), ('USDBWP', 12.755638), ('USDBYN', 2.524225), ('USDBYR', 19600), ('USDBZD', 2.01584), ('USDCAD', 1.353395), ('USDCDF', 2029.99989), ('USDCHF', 0.92395), ('USDCLF', 0.03075), ('USDCLP', 848.250029), ('USDCNY', 6.897897), ('USDCNH', 6.92034), ('USDCOP', 4848.67), ('USDCRC', 591.730443), ('USDCU

KeyError: 'num_code'